# Event Graph to ASM Translation

This notebook demonstrates the translation of Event Graph models to Abstract State Machines (ASM) as implemented in SimASM.

**Reference:** Based on the formal translation algorithm in Section 6 of the paper.

## Overview

1. **Load Event Graph JSON** - Pure Event Graph specification (no SimASM syntax)
2. **Convert to SimASM** - Apply the translation algorithm
3. **Run Little's Law Verification** - Empirically verify correctness
4. **Analyze Results** - Confirm $L = \lambda W$

In [1]:
# Install simasm (uncomment in Colab)
# !pip install simasm

import json
import simasm
from simasm.converter import EventGraphSpec, convert_event_graph

print(f"SimASM version: {simasm.__version__}")

SimASM version: 0.2.2


## 1. Event Graph Specification (JSON)

The Event Graph is specified in a pure JSON format using abstract operations.
This format contains **no SimASM syntax** - all code generation happens in the translator.

### M/M/5 Queue Event Graph

- **Events:** Arrive, AttemptToStart, Start, Finish
- **Parameters:** $\lambda = 0.8$ (arrival rate), $\mu = 1.0$ (service rate), $N = 5$ (servers)

In [2]:
# M/M/5 Queue Event Graph Specification
mm5_event_graph_json = {
    "model_name": "mm5_eg",
    "description": "M/M/5 Queue using Event Graph formalism\nBased on Schruben (1983) Event Graph methodology",
    
    "resources": {
        "generator": {
            "type": "Generator",
            "lists": ["arrivals"],
            "counters": ["arrival_count"]
        },
        "queue": {
            "type": "Queue",
            "lists": ["queues"],
            "counters": ["queue_count"]
        },
        "server": {
            "type": "Server",
            "lists": ["serves", "departures"],
            "counters": ["service_count", "departure_count"],
            "accumulators": ["total_time_in_system", "total_time_in_queue"]
        }
    },
    
    "entities": {
        "Load": {
            "name": "Load",
            "parent": "Object",
            "attributes": {
                "arrival_time": "Real",
                "service_start_time": "Real",
                "departure_time": "Real"
            }
        }
    },
    
    "parameters": {
        "service_capacity": {"type": "Nat", "value": 5, "description": "Number of servers"},
        "iat_mean": {"type": "Real", "value": 1.25, "description": "Inter-arrival time mean (1/lambda)"},
        "ist_mean": {"type": "Real", "value": 1.0, "description": "Service time mean (1/mu)"},
        "sim_end_time": {"type": "Real", "value": 1000.0, "description": "Simulation end time"}
    },
    
    "state_variables": {
        "load_id_counter": {"type": "Nat", "initial": 0}
    },
    
    "random_streams": {
        "interarrival_time": {
            "distribution": "exponential",
            "params": {"mean": "iat_mean"},
            "stream_name": "arrivals"
        },
        "service_time": {
            "distribution": "exponential",
            "params": {"mean": "ist_mean"},
            "stream_name": "service"
        }
    },
    
    "vertices": [
        {
            "name": "Arrive",
            "parameters": [],
            "state_changes": [
                {"op": "increment", "var": "load_id_counter"},
                {"op": "create_entity", "entity_type": "Load", "as_var": "load"},
                {"op": "set_attribute", "entity": "load", "attribute": "id", "value": "load_id_counter"},
                {"op": "set_attribute", "entity": "load", "attribute": "arrival_time", "value": "sim_clocktime"},
                {"op": "add_to_list", "list_name": "arrivals", "resource": "generator", "entity": "load"},
                {"op": "increment_counter", "counter": "arrival_count", "resource": "generator"}
            ],
            "description": "Customer arrival event"
        },
        {
            "name": "AttemptToStart",
            "parameters": [{"load": "Load"}],
            "state_changes": [
                {"op": "add_to_list", "list_name": "queues", "resource": "queue", "entity": "load"},
                {"op": "increment_counter", "counter": "queue_count", "resource": "queue"}
            ],
            "description": "Attempt to start service event"
        },
        {
            "name": "Start",
            "parameters": [{"load": "Load"}],
            "state_changes": [
                {"op": "remove_from_list", "list_name": "queues", "resource": "queue", "entity": "load"},
                {"op": "decrement_counter", "counter": "queue_count", "resource": "queue"},
                {"op": "set_attribute", "entity": "load", "attribute": "service_start_time", "value": "sim_clocktime"},
                {"op": "add_to_list", "list_name": "serves", "resource": "server", "entity": "load"},
                {"op": "increment_counter", "counter": "service_count", "resource": "server"}
            ],
            "description": "Service start event"
        },
        {
            "name": "Finish",
            "parameters": [{"load": "Load"}],
            "state_changes": [
                {"op": "remove_from_list", "list_name": "serves", "resource": "server", "entity": "load"},
                {"op": "decrement_counter", "counter": "service_count", "resource": "server"},
                {"op": "set_attribute", "entity": "load", "attribute": "departure_time", "value": "sim_clocktime"},
                {"op": "compute", "var": "time_in_system", "expression": "sim_clocktime - arrival_time(load)"},
                {"op": "compute", "var": "time_in_queue", "expression": "service_start_time(load) - arrival_time(load)"},
                {"op": "accumulate", "accumulator": "total_time_in_system", "resource": "server", "value": "time_in_system"},
                {"op": "accumulate", "accumulator": "total_time_in_queue", "resource": "server", "value": "time_in_queue"},
                {"op": "add_to_list", "list_name": "departures", "resource": "server", "entity": "load"},
                {"op": "increment_counter", "counter": "departure_count", "resource": "server"}
            ],
            "description": "Service completion event"
        }
    ],
    
    "scheduling_edges": [
        {
            "from": "Arrive",
            "to": "AttemptToStart",
            "delay": 0,
            "condition": "true",
            "priority": 0,
            "parameters": [{"type": "entity", "var": "load"}],
            "comment": "Schedule AttemptToStart for this load (immediate)"
        },
        {
            "from": "Arrive",
            "to": "Arrive",
            "delay": "interarrival_time",
            "condition": "true",
            "priority": 0,
            "parameters": [],
            "comment": "Schedule next arrival"
        },
        {
            "from": "AttemptToStart",
            "to": "Start",
            "delay": 0,
            "condition": {
                "type": "and",
                "conditions": [
                    {"type": "list_length", "list_name": "serves", "resource": "server", "operator": "<", "value": "service_capacity"},
                    {"type": "list_length", "list_name": "queues", "resource": "queue", "operator": ">", "value": 0}
                ]
            },
            "priority": 0,
            "parameters": [{"type": "first_from_list", "list_name": "queues", "resource": "queue"}],
            "comment": "Schedule Start if server available"
        },
        {
            "from": "Start",
            "to": "Finish",
            "delay": "service_time",
            "condition": "true",
            "priority": 0,
            "parameters": [{"type": "entity", "var": "load"}],
            "comment": "Schedule Finish after service time"
        },
        {
            "from": "Finish",
            "to": "Start",
            "delay": 0,
            "condition": {"type": "list_length", "list_name": "queues", "resource": "queue", "operator": ">", "value": 0},
            "priority": 0,
            "parameters": [{"type": "first_from_list", "list_name": "queues", "resource": "queue"}],
            "comment": "Schedule Start for next load if queue not empty"
        }
    ],
    
    "cancelling_edges": [],
    
    "initial_events": [
        {
            "event": "Arrive",
            "time": "interarrival_time",
            "parameters": [],
            "comment": "Schedule first arrival after first interarrival time"
        }
    ],
    
    "stopping_condition": "time_limit",
    
    "observables": {
        "get_queue_count": {
            "name": "get_queue_count",
            "return_type": "Nat",
            "expression": "queue_count(queue)",
            "description": "Number in queue"
        },
        "get_servers_busy": {
            "name": "get_servers_busy",
            "return_type": "Nat",
            "expression": "service_count(server)",
            "description": "Number of busy servers"
        },
        "in_system": {
            "name": "in_system",
            "return_type": "Nat",
            "expression": "get_queue_count() + get_servers_busy()",
            "description": "Total in system"
        }
    },
    
    "predicates": {
        "queue_predicates": {
            "prefix": "queue_eq_",
            "observable": "get_queue_count",
            "values": [0, 1, 2, 3, 4, 5],
            "ge_value": 6
        },
        "busy_predicates": {
            "prefix": "busy_eq_",
            "observable": "get_servers_busy",
            "values": [0, 1, 2, 3, 4, 5]
        },
        "insys_predicates": {
            "prefix": "insys_eq_",
            "observable": "in_system",
            "values": [0, 1, 2, 3, 4, 5],
            "ge_value": 6
        }
    },
    
    "statistics": [
        {
            "name": "L_queue",
            "type": "time_average",
            "observable": "get_queue_count",
            "description": "Average queue length"
        },
        {
            "name": "L_service",
            "type": "time_average",
            "observable": "get_servers_busy",
            "description": "Average number in service"
        },
        {
            "name": "L_system",
            "type": "time_average",
            "observable": "in_system",
            "description": "Average number in system"
        }
    ]
}

print("Event Graph vertices:", [v["name"] for v in mm5_event_graph_json["vertices"]])
print("Scheduling edges:", len(mm5_event_graph_json["scheduling_edges"]))
print("Cancelling edges:", len(mm5_event_graph_json["cancelling_edges"]))

Event Graph vertices: ['Arrive', 'AttemptToStart', 'Start', 'Finish']
Scheduling edges: 5
Cancelling edges: 0


## 2. Translation to SimASM

The `convert_event_graph()` function implements Algorithm 1 from the paper:

1. Construct signature $\Sigma$ (domains, functions)
2. Construct initial state $I_S$
3. Construct initialization rule $P_{Init}$
4. Construct timing rule $P_T$
5. For each vertex $v$, construct event rule $P_{E,v}$
6. Construct dispatcher rule $P_D$
7. Construct run rule $P_R$
8. Construct main rule $P_M$

In [3]:
# Parse JSON to EventGraphSpec
spec = EventGraphSpec(**mm5_event_graph_json)

# Validate the specification
errors = spec.validate_graph()
if errors:
    print("Validation errors:")
    for e in errors:
        print(f"  - {e}")
else:
    print("Specification validated successfully!")

Specification validated successfully!


In [4]:
# Convert Event Graph to SimASM
simasm_code = convert_event_graph(spec)

print(f"Generated {len(simasm_code.splitlines())} lines of SimASM code")
print("\n" + "="*70)
print("Generated SimASM Code (first 100 lines):")
print("="*70)
for i, line in enumerate(simasm_code.splitlines()[:100], 1):
    print(f"{i:4d} | {line}")
print("...")

Generated 310 lines of SimASM code

Generated SimASM Code (first 100 lines):
   1 | // =============================================================================
   2 | // Event Graph Model: mm5_eg - SimASM
   3 | // =============================================================================
   4 | // M/M/5 Queue using Event Graph formalism
   5 | // Based on Schruben (1983) Event Graph methodology
   6 | // Generated: 2026-01-05 20:40:12
   7 | 
   8 | // =============================================================================
   9 | // IMPORT LIBRARIES
  10 | // =============================================================================
  11 | 
  12 | import Random as rnd
  13 | import Stdlib as lib
  14 | 
  15 | // =============================================================================
  16 | // DOMAIN DECLARATIONS
  17 | // =============================================================================
  18 | 
  19 | domain Object
  20 | domain Generator <: Object


In [5]:
# Save generated code to file
output_path = "mm5_eg_translated.simasm"
with open(output_path, "w") as f:
    f.write(simasm_code)
print(f"Saved to: {output_path}")

Saved to: mm5_eg_translated.simasm


## 3. Run Little's Law Experiment

Little's Law states: $L = \lambda W$

Where:
- $L$ = average number of jobs in the system
- $\lambda$ = arrival rate (1/1.25 = 0.8)
- $W$ = average time in the system

For M/M/5 with $\lambda = 0.8$, $\mu = 1.0$, $N = 5$:
- Traffic intensity: $a = \lambda/\mu = 0.8$
- Server utilization: $\rho = a/N = 0.16$

In [6]:
%%simasm experiment
// =============================================================================
// Little's Law Verification - Translated Event Graph M/M/5 Queue
// =============================================================================

experiment LittlesLawMM5:
    model := "mm5_eg_translated.simasm"
    
    replication:
        count: 10
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 12345
    endreplication
    
    statistics:
        // L: Average number in system
        stat L_system: time_average
            expression: "lib.length(queues(queue)) + lib.length(serves(server))"
        endstat
        
        // L_q: Average number in queue
        stat L_queue: time_average
            expression: "lib.length(queues(queue))"
        endstat
        
        // L_s: Average number in service
        stat L_service: time_average
            expression: "lib.length(serves(server))"
        endstat

        // W: Average time in system
        stat W_system: count
            expression: "total_time_in_system(server) / departure_count(server)"
        endstat

        // W_q: Average time in queue
        stat W_queue: count
            expression: "total_time_in_queue(server) / departure_count(server)"
        endstat
        
        // Total arrivals (for computing lambda)
        stat total_arrivals: count
            expression: "arrival_count(generator)"
        endstat

        // Total departures
        stat total_departures: count
            expression: "departure_count(server)"
        endstat
        
        // Server utilization
        stat rho_utilization: utilization
            expression: "lib.length(serves(server)) > 0"
        endstat

        // Average servers busy
        stat avg_servers_busy: time_average
            expression: "lib.length(serves(server))"
        endstat
        
    endstatistics
    
    output:
        format: "json"
        file_path: "littles_law_mm5_results.json"
    endoutput
endexperiment

  Replication 10/10...


Statistic,Mean,Std Dev,95% CI
L_system,0.8072,0.0461,"[0.7742, 0.8402]"
L_queue,0.0003,0.0006,"[-0.0001, 0.0007]"
L_service,0.8069,0.0460,"[0.7740, 0.8398]"
W_system,1.0137,0.0379,"[0.9866, 1.0408]"
W_queue,0.0004,0.0006,"[-0.0000, 0.0009]"
total_arrivals,797.8000,23.3371,"[781.1068, 814.4932]"
total_departures,796.8000,23.3895,"[780.0694, 813.5306]"
rho_utilization,0.5535,0.0191,"[0.5398, 0.5671]"
avg_servers_busy,0.8069,0.0460,"[0.7740, 0.8398]"


ExperimentResult(config=ExperimentConfig(name='LittlesLawMM5', model_path='c:\\Users\\steve\\simasm\\simasm\\notebooks\\mm5_eg_translated.simasm', model_source=None, replications=ReplicationSettings(count=10, warmup=100.0, length=1000.0, base_seed=12345), statistics=[StatisticConfig(name='L_system', type='time_average', domain=None, expr='lib.length(queues(queue)) + lib.length(serves(server))', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_queue', type='time_average', domain=None, expr='lib.length(queues(queue))', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_service', type='time_average', domain=None, expr='lib.length(serves(server))', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='W_system', type='count', domain=None, expr='total_time_i

## 4. Verify Little's Law

We verify that the translation is correct by checking Little's Law: $L = \lambda W$

In [ ]:
# Load results from the experiment output
import json

with open("littles_law_mm5_results.json") as f:
    results = json.load(f)

# Theoretical values for M/M/5 (lambda=0.8, mu=1.0, N=5)
lambda_rate = 0.8  # arrival rate = 1/1.25
mu_rate = 1.0      # service rate = 1/1.0
N = 5              # number of servers
rho_theoretical = lambda_rate / (N * mu_rate)  # 0.16

# Get simulated values from summary statistics
summary = results.get("summary", {})
L_simulated = summary.get("L_system", {}).get("mean", 0)
W_simulated = summary.get("W_system", {}).get("mean", 0)
avg_busy = summary.get("avg_servers_busy", {}).get("mean", 0)

# Calculate Little's Law verification
L_from_littles_law = lambda_rate * W_simulated

print("="*70)
print("Little's Law Verification: L = lambda * W")
print("="*70)
print(f"")
print(f"Parameters:")
print(f"  lambda (arrival rate)     = {lambda_rate}")
print(f"  mu (service rate)         = {mu_rate}")
print(f"  N (servers)               = {N}")
print(f"  rho (theoretical)         = {rho_theoretical:.4f}")
print(f"")
print(f"Simulated Values:")
print(f"  L (avg in system)         = {L_simulated:.4f}")
print(f"  W (avg time in system)    = {W_simulated:.4f}")
print(f"  Avg servers busy          = {avg_busy:.4f}")
print(f"")
print(f"Little's Law Check:")
print(f"  L (simulated)             = {L_simulated:.4f}")
print(f"  lambda * W                = {L_from_littles_law:.4f}")
print(f"  Difference                = {abs(L_simulated - L_from_littles_law):.4f}")
if L_simulated > 0:
    print(f"  Relative Error            = {abs(L_simulated - L_from_littles_law) / L_simulated * 100:.2f}%")
print(f"")

if L_simulated > 0 and abs(L_simulated - L_from_littles_law) / L_simulated < 0.05:
    print("VERIFICATION PASSED: Little's Law holds within 5% tolerance")
else:
    print("VERIFICATION WARNING: Check results (may need more replications)")

## 5. Summary

This notebook demonstrated:

1. **Pure Event Graph JSON** - The Event Graph is specified using abstract operations without any SimASM syntax
2. **Automatic Translation** - The `convert_event_graph()` function implements Algorithm 1 to generate SimASM code
3. **Correctness Verification** - Little's Law ($L = \lambda W$) confirms the translation preserves the behavioral semantics

### Generated ASM Components

| Event Graph Element | ASM Construct |
|---------------------|---------------|
| Vertices {Arrive, AttemptToStart, Start, Finish} | Event rules $P_{E,v}$ |
| Scheduling edges | FEL insertions |
| Initial events | Initialization rule $P_{Init}$ |
| Next-event mechanism | Timing rule $P_T$ + Dispatcher $P_D$ |

In [ ]:
# Clean up
%simasm_clear
print("Done.")